# Certified order-2 optima, corrected pipeline (v2)

Regenerates Table 3 of the FOTD order-2 paper with one consistent protocol, fixing two defects
of the original pipeline:

1. **Gain convention.** The velocity-form realization u[k] = (1+z2)u[k-1] - z2 u[k-2]
   + ki dt e[k] + kp De + (kd/dt) D2e is exactly the backward-Euler discretization of the
   continuous filtered PID C(s) = (Kd s^2 + Kp s + Ki)/(s(1+s/N)) with raw weights
   kp_raw = Kp(1-z2), ki_raw = Ki(1-z2), kd_raw = Kd(1-z2) and z2 = 1/(1+dt N).
   The old table reported raw weights as if they were PID gains.
2. **Rate consistency.** The old fine re-evaluation matched the pole (z2 -> z2^(FDT/dt)) but
   not the (1-z2) gain scale, so the fine run measured a controller roughly 4x hotter than
   the trained one. Here the search variable IS the continuous (Kp, Ki, Kd, N); every rate
   discretizes the same object, so dt = 0.025 and dt = 0.005 measure the same controller.

Protocol: zero-tolerance search at dt = 0.025 (cross-seeded, fixpoint sweeps), then
verification of every winner at dt = 0.005. Output: corrected table with both the PID gains
and the raw velocity weights (traceability to the old table), LaTeX rows, JSON, and a MATLAB
check line. Runtime roughly 20 to 40 min. **Runtime type: CPU.**

In [ ]:
import numpy as np, json, time
import matplotlib.pyplot as plt

K, L, BAND = 1.0, 1.0, 0.02
DT_C, DT_F = 0.025, 0.005
TL_GRID = [0.2,0.3,0.5,0.7,1.0,1.5,2.0,3.0,4.0,5.0]

def sim_o2(Kp, Ki, Kd, N, T, h, horizon=None):
    """Vectorized exact discrete loop. Kp,Ki,Kd,N arrays (G,), scalars T,h.
    Controller: backward-Euler of (Kd s^2 + Kp s + Ki)/(s(1+s/N)).
    Plant: exact ZOH of K e^{-Ls}/(Ts+1). Returns ts, tv0, ymax, tvu."""
    Kp,Ki,Kd,N = [np.atleast_1d(np.asarray(v,float)) for v in (Kp,Ki,Kd,N)]
    G = len(Kp)
    if horizon is None: horizon = 10.0*T + 20.0*L
    nstep = int(round(horizon/h)); dsamp = int(round(L/h))
    z2 = 1.0/(1.0+h*N)
    d1, d2 = 1.0+z2, -z2
    ci, cp, cd = Ki*(1.0-z2)*h, Kp*(1.0-z2), Kd*(1.0-z2)/h
    aa = np.exp(-h/T); bb = K*(1.0-aa)
    ub = np.zeros((nstep, G)); y = np.zeros(G)
    e1=np.zeros(G); e2=np.zeros(G); u1=np.zeros(G); u2=np.zeros(G)
    tv0=np.zeros(G); ymax=np.zeros(G); ts=np.zeros(G); tvu=np.zeros(G)
    for k in range(nstep):
        e = 1.0 - y
        u = d1*u1 + d2*u2 + ci*e + cp*(e-e1) + cd*(e-2.0*e1+e2)
        if k > 0: tvu += np.abs(u-u1)
        ub[k] = u
        ud = ub[k-dsamp] if k >= dsamp else np.zeros(G)
        yn = aa*y + bb*ud
        tv0 += np.maximum(0.0, -(yn-y)); y = yn; ymax = np.maximum(ymax, y)
        ts = np.where(np.abs(1.0-y) > BAND, (k+1)*h, ts)
        e2, e1 = e1, e; u2, u1 = u1, u
    return ts, tv0, ymax, tvu, nstep*h

def sim_pi(Kp, Ki, T, h, horizon=None):
    Kp,Ki = np.atleast_1d(np.asarray(Kp,float)), np.atleast_1d(np.asarray(Ki,float))
    G = len(Kp)
    if horizon is None: horizon = 10.0*T + 20.0*L
    nstep = int(round(horizon/h)); dsamp = int(round(L/h))
    aa = np.exp(-h/T); bb = K*(1.0-aa)
    q0, q1 = Kp + Ki*h, -Kp
    ub = np.zeros((nstep,G)); y=np.zeros(G); e1=np.zeros(G); u1=np.zeros(G)
    tv0=np.zeros(G); ymax=np.zeros(G); ts=np.zeros(G); tvu=np.zeros(G)
    for k in range(nstep):
        e = 1.0 - y
        u = u1 + q0*e + q1*e1
        if k > 0: tvu += np.abs(u-u1)
        ub[k] = u
        ud = ub[k-dsamp] if k >= dsamp else np.zeros(G)
        yn = aa*y + bb*ud
        tv0 += np.maximum(0.0, -(yn-y)); y = yn; ymax = np.maximum(ymax, y)
        ts = np.where(np.abs(1.0-y) > BAND, (k+1)*h, ts)
        e1, u1 = e, u
    return ts, tv0, ymax, tvu, nstep*h

def raw_from_pid(Kp, Ki, Kd, N, h):
    z2 = 1.0/(1.0+h*N)
    return Kp*(1.0-z2), Ki*(1.0-z2), Kd*(1.0-z2), z2

def pid_from_raw(kp_r, ki_r, kd_r, z2, h):
    g = 1.0/(1.0-z2); N = (1.0-z2)/(h*z2)
    return kp_r*g, ki_r*g, kd_r*g, N

# sanity: the two internally consistent rows of the OLD table must replay at dt=0.025
OLD = {2.0:(0.6834,0.2972,0.1836,0.500,3.200), 5.0:(1.6115,0.3029,0.4852,0.494,3.200)}
for tl,(kp_r,ki_r,kd_r,z2,ts_tab) in OLD.items():
    Kp,Ki,Kd,N = pid_from_raw(kp_r,ki_r,kd_r,z2,DT_C)
    ts,tv0,ym,_,_ = sim_o2(Kp,Ki,Kd,N,tl*L,DT_C,horizon=40*L)
    print(f'replay T/L={tl}: PID (Kp={Kp:.4f} Ki={Ki:.4f} Kd={Kd:.4f} N={N:.1f}) '
          f'-> Ts/L={ts[0]/L:.3f} (old {ts_tab})  TV0={tv0[0]:.1e}  OK={abs(ts[0]/L-ts_tab)<1e-9}')

## 1. Exact PI ceiling at both rates

The order-2 class contains PI, so the PI optimum both seeds the search and lower-bounds
nothing: every certified order-2 Ts must come out at or below the PI ceiling of its plant.
The fine T/L = 2 point self-verifies against the published tangency anchor
Kp = 0.906, Ki = 0.4265, Ts/L = 4.81.

In [ ]:
def grid_pi(TL, h, G=40, rounds=5, box=None):
    T = TL*L
    if box is None: lo_p,hi_p,lo_i,hi_i = 0.02, 4.0*max(TL,0.5), 0.02, 1.2
    else: lo_p,hi_p,lo_i,hi_i = box
    best = None
    for _ in range(rounds):
        gp, gi = np.linspace(lo_p,hi_p,G), np.linspace(lo_i,hi_i,G)
        Kp, Ki = [v.ravel() for v in np.meshgrid(gp,gi,indexing='ij')]
        ts,tv0,ym,tvu,hz = sim_pi(Kp,Ki,T,h)
        feas = (tv0<=1e-9)&(ym<=1+1e-9)&(ts>0)&(ts<hz-2*h)
        if not feas.any(): return None
        j = int(np.argmin(np.where(feas,ts,1e9)))
        best = (Kp[j],Ki[j],ts[j],tvu[j])
        dp,di = (hi_p-lo_p)/(G-1)*2, (hi_i-lo_i)/(G-1)*2
        lo_p,hi_p = max(1e-3,Kp[j]-dp), Kp[j]+dp
        lo_i,hi_i = max(1e-3,Ki[j]-di), Ki[j]+di
    return best

t0 = time.time()
PIC, PIF = {}, {}
for tl in TL_GRID:
    PIC[tl] = grid_pi(tl, DT_C)
    PIF[tl] = grid_pi(tl, DT_F, box=(0.5*PIC[tl][0],1.5*PIC[tl][0],0.5*PIC[tl][1],1.5*PIC[tl][1]))
    print(f'T/L={tl:4.1f}  coarse: Kp={PIC[tl][0]:.4f} Ki={PIC[tl][1]:.4f} Ts/L={PIC[tl][2]:.3f}   '
          f'fine: Kp={PIF[tl][0]:.4f} Ki={PIF[tl][1]:.4f} Ts/L={PIF[tl][2]:.3f}  ({time.time()-t0:.0f}s)', flush=True)
print('\nanchors: coarse T/L=1 should be ~4.400; fine T/L=2 should be ~4.81 with Kp~0.906 Ki~0.4265')

## 2. Zero-tolerance order-2 search at dt = 0.025

Search variables: continuous (Kp, Ki, Kd, N). Feasibility: TV0 <= 1e-9 and no overshoot on
the coarse grid. Cross-plant seeding (Kp, Kd scale with T; Ki, N roughly invariant) with
fixpoint sweeps, the mechanism validated in the frontier notebook. The PI optimum is always
in the seed pool, so Ts2 <= TsPI is guaranteed by construction.

In [ ]:
def solve_plant(tl, seeds, rounds=4, top=3):
    """min Ts s.t. zero-tolerance feasibility at DT_C. Returns (Ts, TVu, (Kp,Ki,Kd,N))."""
    T = tl*L; best = None
    # evaluate all seeds as incumbents
    S = np.array(seeds, float)
    ts,tv0,ym,tvu,hz = sim_o2(S[:,0],S[:,1],S[:,2],S[:,3],T,DT_C)
    feas = (tv0<=1e-9)&(ym<=1+1e-9)&(ts>0)&(ts<hz-2*DT_C)
    order = np.argsort(np.where(feas, ts, 1e9))
    incumbents = [tuple(S[j]) for j in order[:top] if feas[j]]
    if not incumbents: incumbents = [tuple(S[order[0]])]
    if feas.any():
        j = order[0]; best = (ts[j], tvu[j], tuple(S[j]))
    for inc in incumbents:
        kp0, ki0, kd0, N0 = inc
        box = dict(kp=(0.3*kp0, 2.0*kp0, 10), ki=(0.3*ki0, 2.0*ki0, 10),
                   kd=(0.0, max(2.0*kd0, 0.15), 8), N=(5.0, 90.0, 8))
        for rnd in range(rounds):
            kps=np.linspace(*box['kp']); kis=np.linspace(*box['ki'])
            kds=np.linspace(*box['kd']); Ns =np.linspace(*box['N'])
            KP,KI,KD,NN = [v.ravel() for v in np.meshgrid(kps,kis,kds,Ns,indexing='ij')]
            ts,tv0,ym,tvu,hz = sim_o2(KP,KI,KD,NN,T,DT_C)
            feas = (tv0<=1e-9)&(ym<=1+1e-9)&(ts>0)&(ts<hz-2*DT_C)
            if not feas.any(): break
            j = int(np.argmin(np.where(feas,ts,1e9)))
            cand = (ts[j],tvu[j],(KP[j],KI[j],KD[j],NN[j]))
            if best is None or cand[0] < best[0]-1e-12: best = cand
            kp_,ki_,kd_,N_ = cand[2]
            dp=(box['kp'][1]-box['kp'][0])/9; di=(box['ki'][1]-box['ki'][0])/9
            dd=(box['kd'][1]-box['kd'][0])/7; dn=(box['N'][1]-box['N'][0])/7
            box=dict(kp=(max(1e-3,kp_-1.5*dp),kp_+1.5*dp,8), ki=(max(1e-3,ki_-1.5*di),ki_+1.5*di,8),
                     kd=(max(0.0,kd_-1.5*dd),kd_+1.5*dd,7), N=(max(2.0,N_-1.5*dn),min(200.0,N_+1.5*dn),7))
    return best

def scaled(c, tl_from, tl_to):
    s = tl_to/tl_from
    return (c[0]*s, c[1], c[2]*s, c[3])

win = {}
t0 = time.time()
for sweep in range(3):
    improved = 0
    for tl in TL_GRID:
        pool = [(PIC[tl][0], PIC[tl][1], 0.0, 40.0),
                (PIC[tl][0], PIC[tl][1], 0.02*tl, 40.0)]
        for tl2, w in win.items():
            pool.append(scaled(w[2], tl2, tl) if tl2 != tl else w[2])
        m = solve_plant(tl, pool)
        if m is None:
            print(f'  sweep {sweep} T/L={tl}: NO FEASIBLE POINT'); continue
        old = win.get(tl)
        if old is None or m[0] < old[0]-1e-12:
            win[tl] = m; improved += 1
        print(f'  sweep {sweep} T/L={tl:4.1f}  Ts/L={win[tl][0]/L:.3f}  ({time.time()-t0:.0f}s)', flush=True)
    print(f'sweep {sweep}: {improved} plants improved')
    if improved == 0: break

## 3. Fine verification and the corrected table

Every coarse winner is re-simulated at dt = 0.005 as the SAME continuous controller. A row
is verified when the fine run is monotone (TV0 <= 1e-6) with no overshoot and its settling
agrees with the coarse value to within a few grid steps. If a fine run shows a small
violation, gains are backed off by a scalar gamma until it clears, and both values are
reported. The table also prints the raw velocity weights for traceability to the old Table 3.

In [ ]:
rows = []
print(f"{'T/L':>4} | {'Kp':>7} {'Ki':>7} {'Kd':>7} {'N':>6} | {'Ts_c/L':>7} {'Ts_f/L':>7} {'TsPI/L':>7} {'gain%':>6} | "
      f"{'TV0_f':>8} {'OS_f':>8} {'gam':>5} | raw (kp ki kd z2)")
print('-'*130)
for tl in TL_GRID:
    if tl not in win: continue
    ts_c, tvu_c, (Kp,Ki,Kd,N) = win[tl]
    gam = 1.0
    for _ in range(30):
        ts_f,tv0_f,ym_f,tvu_f,_ = sim_o2(gam*Kp,gam*Ki,gam*Kd,N,tl*L,DT_F)
        if tv0_f[0] <= 1e-6 and ym_f[0] <= 1+1e-6: break
        gam *= 0.995
    Kpv,Kiv,Kdv = gam*Kp, gam*Ki, gam*Kd
    ts_c2,_,_,_,_ = sim_o2(Kpv,Kiv,Kdv,N,tl*L,DT_C)
    kp_r,ki_r,kd_r,z2 = raw_from_pid(Kpv,Kiv,Kdv,N,DT_C)
    tsPI = PIF[tl][2]
    g = 100*(tsPI-ts_f[0])/tsPI
    rows.append(dict(TL=tl, Kp=Kpv, Ki=Kiv, Kd=Kdv, N=N, z2=z2,
                     kp_raw=kp_r, ki_raw=ki_r, kd_raw=kd_r,
                     Ts_coarse=ts_c2[0]/L, Ts_fine=ts_f[0]/L, TsPI_fine=tsPI/L,
                     gain_pct=g, TV0_fine=float(tv0_f[0]), OS_fine=float(ym_f[0]-1), gamma=gam))
    print(f'{tl:4.1f} | {Kpv:7.4f} {Kiv:7.4f} {Kdv:7.4f} {N:6.1f} | {ts_c2[0]/L:7.3f} {ts_f[0]/L:7.3f} '
          f'{tsPI/L:7.3f} {g:6.1f} | {tv0_f[0]:8.1e} {ym_f[0]-1:8.1e} {gam:5.3f} | '
          f'({kp_r:.4f} {ki_r:.4f} {kd_r:.4f} {z2:.3f})')

assert all(r['Ts_fine'] <= r['TsPI_fine'] + 1e-9 for r in rows), 'order-2 must not lose to PI'
json.dump(rows, open('order2_certified_v2.json','w'), indent=1)
print('\nsaved order2_certified_v2.json  <- download this')

print('\nLaTeX rows (T/L & Kp & Ki & Kd & N & Ts/L & Ts_PI/L):')
for r in rows:
    print(f"{r['TL']:.1f} & {r['Kp']:.4f} & {r['Ki']:.4f} & {r['Kd']:.3f} & {r['N']:.0f} & "
          f"{r['Ts_fine']:.3f} & {r['TsPI_fine']:.3f} \\\\")

r1 = [r for r in rows if r['TL']==1.0][0]
print('\nMATLAB check (T/L=1), expect SettlingTime within a few percent of Ts_fine:')
print(f"s=tf('s'); Kp={r1['Kp']:.4f}; Ki={r1['Ki']:.4f}; Kd={r1['Kd']:.4f}; N={r1['N']:.2f};")
print("C=(Kd*s^2+Kp*s+Ki)/(s*(1+s/N)); P=exp(-s)/(s+1);")
print("stepinfo(feedback(C*P,1))")

## 4. Structure of the corrected optima: cancellation and the reduced loop

The pole-zero collapse claim is scale invariant (numerator roots do not see the 1/(1-z2)
factor), but the reduced-loop constants do. This cell reports the corrected constants:
slow-zero mismatch against the plant pole, the fast zero tau2, the integral gain, and the
filter, all in delay units.

In [ ]:
print(f"{'T/L':>4} {'slow zero':>10} {'1/T':>7} {'mismatch%':>9} {'tau2/L':>7} {'Ki*L':>7} {'N*L':>6} {'Ts_f/L':>7}")
for r in rows:
    Kp,Ki,Kd,N,tl = r['Kp'],r['Ki'],r['Kd'],r['N'],r['TL']
    if Kd > 1e-9:
        rts = np.roots([Kd,Kp,Ki])
        rts = np.sort(np.abs(rts[np.isreal(rts)].real)) if np.isreal(rts).all() else np.abs(rts)
        slow, fast = rts[0], rts[-1]
        mism = 100*(slow - 1.0/tl)/(1.0/tl)
        print(f'{tl:4.1f} {slow:10.4f} {1.0/tl:7.4f} {mism:9.2f} {1.0/fast:7.4f} {Ki:7.4f} {N:6.1f} {r["Ts_fine"]:7.3f}')
    else:
        print(f'{tl:4.1f}   (Kd ~ 0, PI-like)')
walls = [r['Ts_fine'] for r in rows]
print(f'\nwall: min {min(walls):.3f}  max {max(walls):.3f}  spread {max(walls)-min(walls):.3f} L')
print('flat-wall claim holds if the spread is a few grid steps of the coarse search (0.025 L each).')

---
## Download before closing

```python
from google.colab import files
files.download('order2_certified_v2.json')
```

**Reading the result.** Section 0 replay lines must both print OK=True, tying the new pipeline
to the two internally consistent rows of the old table. Section 1 anchors the PI ceiling at
both rates against published values. Section 3 is the corrected Table 3: gains are true
filtered-PID gains, coarse and fine settling agree because they discretize the same
controller, and gamma = 1.000 everywhere means no fine backoff was needed. Section 4 gives
the corrected open-problem constants; expect Ki*L near 0.6 rather than the old 0.30, N*L
near 40 rather than 28, and the cancellation mismatch small for T/L >= 0.5. Paste all cell
outputs back for the paper correction.